# VoxSync Colab (online)

Transcribe video o YouTube, traduce a espanol **sin Google** (modelo local) y genera:
- SRT en espanol
- Video MP4 con subtitulos quemados (voces originales + texto en ES)

1. Entorno de ejecucion > Cambiar tipo de entorno > GPU (T4)
2. Ejecuta las celdas en orden
3. Descarga el SRT y/o el video al final


## 1. Instalar dependencias


In [ ]:
!pip install -q openai-whisper yt-dlp transformers sentencepiece sacremoses
!apt -y -qq install ffmpeg
print("OK")


## 2. Configuracion


In [ ]:
YOUTUBE_URL = ""  # ejemplo: "https://www.youtube.com/watch?v=XXXX"
SOURCE_LANG = "en"
TARGET_LANG = "es"
WHISPER_MODEL = "small"  # tiny | base | small | medium
BURN_SUBTITLES = True    # True = genera video con subtitulos encima
print("Config OK")


## 3. Cargar video


In [ ]:
from pathlib import Path
import subprocess

work = Path("/content/voxsync")
work.mkdir(exist_ok=True)

if YOUTUBE_URL.strip():
    print("Descargando YouTube...")
    subprocess.run([
        "yt-dlp", "-f", "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/b",
        "-o", str(work / "%(title)s.%(ext)s"), "--no-playlist", YOUTUBE_URL.strip()
    ], check=True)
    cands = []
    for ext in (".mp4", ".webm", ".mkv", ".m4a", ".mp3"):
        cands.extend(work.glob("*" + ext))
    video_path = max(cands, key=lambda p: p.stat().st_mtime)
    print("Video:", video_path.name)
else:
    from google.colab import files
    print("Sube un video o audio...")
    uploaded = files.upload()
    name = list(uploaded.keys())[0]
    video_path = work / name
    video_path.write_bytes(uploaded[name])
    print("Archivo:", video_path.name)

print("Listo:", video_path)


## 4. Extraer audio


In [ ]:
audio_path = work / "audio.wav"
subprocess.run([
    "ffmpeg", "-y", "-i", str(video_path),
    "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1",
    str(audio_path)
], check=True, capture_output=True)
print("Audio OK:", audio_path)


## 5. Transcribir con Whisper


In [ ]:
import whisper

print("Cargando", WHISPER_MODEL)
model = whisper.load_model(WHISPER_MODEL)
print("Transcribiendo...")
result = model.transcribe(str(audio_path), language=SOURCE_LANG or None, verbose=False)
segments = result["segments"]
print(len(segments), "segmentos")
for s in segments[:3]:
    print(round(s["start"], 1), s["text"][:100])


## 6. Traducir LOCAL (sin Google, sin limite de requests)


In [ ]:
import torch
from transformers import pipeline

print("Cargando traductor local Helsinki-NLP/opus-mt-en-es...")
device = 0 if torch.cuda.is_available() else -1
model_name = "Helsinki-NLP/opus-mt-en-es"
if SOURCE_LANG == "es" and TARGET_LANG == "en":
    model_name = "Helsinki-NLP/opus-mt-es-en"

translator = pipeline("translation", model=model_name, device=device)

texts = [(s.get("text") or "").strip() for s in segments]
translations = []
batch_size = 16
for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    clean = [t if t else "." for t in batch]
    if SOURCE_LANG == TARGET_LANG:
        translations.extend(batch)
    else:
        out = translator(clean, max_length=256)
        for j, item in enumerate(out):
            translations.append(item["translation_text"] if batch[j] else "")
    print(f"  traducidos {min(i + batch_size, len(texts))}/{len(texts)}")

for seg, tr in zip(segments, translations):
    seg["translation"] = tr

print("OK - vista previa:")
for s in segments[:3]:
    print("ORIG:", s["text"][:80])
    print("TRAD:", s["translation"][:80])
    print()


## 7. Generar SRT


In [ ]:
def ts(seconds):
    total = int(float(seconds))
    ms = int((float(seconds) - total) * 1000)
    h, rem = divmod(total, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

base = video_path.stem
srt_path = work / f"{base}.{TARGET_LANG}.srt"
srt_bi = work / f"{base}.bilingual.srt"

with open(srt_path, "w", encoding="utf-8") as f:
    for i, seg in enumerate(segments, 1):
        text = (seg.get("translation") or seg["text"]).strip()
        f.write(f"{i}\n{ts(seg['start'])} --> {ts(seg['end'])}\n{text}\n\n")

with open(srt_bi, "w", encoding="utf-8") as f:
    for i, seg in enumerate(segments, 1):
        orig = seg["text"].strip()
        trans = (seg.get("translation") or orig).strip()
        body = orig + "\n" + trans if trans != orig else orig
        f.write(f"{i}\n{ts(seg['start'])} --> {ts(seg['end'])}\n{body}\n\n")

print("SRT:", srt_path)
print("Bilingual:", srt_bi)


## 8. Video con subtitulos quemados (voces originales + texto ES)


In [ ]:
out_video = None
if BURN_SUBTITLES:
    out_video = work / f"{base}_ES.mp4"
    srt_for_ffmpeg = str(srt_path).replace("\\", "/").replace(":", "\\:")
    print("Generando video con subtitulos...")
    subprocess.run([
        "ffmpeg", "-y", "-i", str(video_path),
        "-vf", f"subtitles={srt_for_ffmpeg}:force_style='FontSize=18,Outline=1'",
        "-c:a", "copy",
        str(out_video)
    ], check=True)
    print("Video listo:", out_video)
else:
    print("BURN_SUBTITLES=False: solo SRT")


## 9. Descargar resultados


In [ ]:
from google.colab import files

files.download(str(srt_path))
files.download(str(srt_bi))
if out_video is not None and out_video.exists():
    files.download(str(out_video))
print("Listo: SRT en espanol + video con voces y subtitulos")


---
## Extra: solo traducir un SRT que ya tengas (sin re-transcribir)

Si ya tenes un `.srt` en ingles, descomenta el codigo de la siguiente celda.


In [ ]:
# OPCIONAL: traducir SRT existente (descomenta para usar)
# from google.colab import files
# import re, torch
# from transformers import pipeline
# from pathlib import Path
# work = Path("/content/voxsync")
# work.mkdir(exist_ok=True)
# print("Sube el SRT en ingles...")
# up = files.upload()
# srt_in = work / list(up.keys())[0]
# srt_in.write_bytes(up[list(up.keys())[0]])
# raw = srt_in.read_text(encoding="utf-8", errors="ignore")
# blocks = re.split(r"\n\s*\n", raw.strip())
# entries = []
# for block in blocks:
#     lines = [l.strip() for l in block.splitlines() if l.strip()]
#     if len(lines) >= 3:
#         entries.append({"idx": lines[0], "timing": lines[1], "text": " ".join(lines[2:])})
# device = 0 if torch.cuda.is_available() else -1
# translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-es", device=device)
# texts = [e["text"] for e in entries]
# translations = []
# for i in range(0, len(texts), 16):
#     batch = texts[i:i+16]
#     clean = [t if t.strip() else "." for t in batch]
#     out = translator(clean, max_length=256)
#     for j, item in enumerate(out):
#         translations.append(item["translation_text"] if batch[j].strip() else "")
#     print(min(i+16, len(texts)), "/", len(texts))
# srt_es = work / "subtitulos_ES_REAL.srt"
# with open(srt_es, "w", encoding="utf-8") as f:
#     for e, tr in zip(entries, translations):
#         f.write(f"{e['idx']}\n{e['timing']}\n{tr}\n\n")
# files.download(str(srt_es))
# print("Listo:", srt_es)
print("Celda opcional lista (descomenta el codigo de arriba si la necesitas)")
